# Differential Gene Expression Analysis

After clustering and annotation identified distinct cell types, we next examined the transcriptional changes induced by our manipulation.
To characterize transcriptional changes associated with the manipulation, two complementary DEG strategies were applied:

1. **Wilcoxon** (cell-level) – A non-parametric, distribution-free test performed across all single cells, suitable for rapid identification of candidate DEG patterns.
2. **MAST** (model-based) –  A hurdle-based generalized linear model that accounts for zero inflation and cellular detection rate, providing statistically rigorous and reproducible DEG validation.

Together, these methods offer both high sensitivity (Wilcoxon) and high specificity (MAST), ensuring that detected gene-expression differences are supported by convergent evidence from independent statistical models.


In [ ]:
import scanpy as sc
import omicverse as ov
import numpy as np
import pandas as pd
import os
import random
import memento
from datetime import date
import matplotlib.pyplot as plt
import seaborn as sns


ov.plot_set(dpi=150,font_path='Arial')

In [ ]:
# set code outputed by R is UTF-8 format
import os
os.environ["R_USER"] = os.getenv("HOME", "")
os.environ["R_HOME"] = "D:/R/R-4.3.3"
os.environ["LANG"] = "en_US.UTF-8"
os.environ["R_ENCODING"] = "UTF-8"

import logging
import anndata2ri
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
ov.settings.cpu_init()

In [ ]:
# adata_sub = adata[adata.obs["sample"].isin(["PC94", "PC110", "PC112", "PC114"])].copy()   #place vs nonplace
# adata_sub = adata[adata.obs["sample"].isin(["PC103", "PC106", "PC118"])].copy() #place vs silent
# adata_sub = adata[adata.obs["sample"].isin(["PC92", "PC105", "PC132"])].copy()  #nonplace vs silent

BASE_DIR = r"F:\Data\snRNAseq\PlaceCell\FilteredData"
INPUT_FILE = f"PCAnalysis_All_QC_20260127.h5ad"

today = date.today().strftime("%Y%m%d")
input_path = os.path.join(BASE_DIR, INPUT_FILE)

In [ ]:
adata = sc.read_h5ad(input_path)
adata

In [ ]:
adata_annotation = sc.read_h5ad(
    r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData\All_QC_filtered_annotation_20260604.h5ad"
)
assert adata.n_obs == adata_annotation.n_obs
assert (adata.obs_names == adata_annotation.obs_names).all()

# 转移 cell type 注释
adata.obs["cell_types"] = adata_annotation.obs["cell_types"].values

# 去掉 Unassigned
adata = adata[adata.obs["cell_types"] == "CA1-ProS"].copy()

In [ ]:
# assert adata.n_obs == adata_annotation.n_obs
# assert (adata.obs_names == adata_annotation.obs_names).all()

# # 转移 cell type 注释
# adata.obs["cell_types"] = adata_annotation.obs["cell_types"].values

# # 去掉 Unassigned
# adata = adata[adata.obs["cell_types"] == "CA1-ProS"].copy()

In [ ]:
adata

In [ ]:
nan_count = np.isnan(adata.X).sum()
print(f"Number of NaN values in adata.X: {nan_count}")
print(f"Max value in martix:" ,adata.X.max())

In [ ]:
adata

In [ ]:
# cell_type = "Astro"

In [ ]:
# cell_subset = adata[adata.obs['cell_types'] == cell_type]
# samples =  ["PC94", "PC110", "PC112", "PC114"]
# samples = ["PC103", "PC106", "PC118"]
# samples = ["PC92", "PC105", "PC132"]

# cell_subset = adata[adata.obs["sample"].isin(samples)].copy()


In [ ]:
cell_subset =adata

In [ ]:
cell_subset.obs['condition'].value_counts()

In [ ]:
cell_subset.obs['cell_types'].value_counts()

---

### 1. Differential Expression Analysis using **Wilcoxon Test**

We performed single-cell–level differential expression analysis using the Wilcoxon rank-sum test implemented in Seurat’s **FindMarkers()** function.
This non-parametric method compares expression distributions between groups without assuming normality.
However, its p-values should be interpreted cautiously, because the test treats each cell as an independent observation and does not model the correlation among cells originating from the same biological sample.
As a result, Wilcoxon tends to report highly significant p-values and is primarily used as an exploratory, high-sensitivity screening approach.

---


In [ ]:
print(cell_subset.layers.keys())
print(cell_subset.X[:10, :10])
print(cell_subset.layers["counts"][:10, :10])
print(cell_subset.raw)

In [ ]:
# Data Preparation
data = cell_subset.layers["counts"].T
genes = np.array(cell_subset.var_names)
cells = np.array(cell_subset.obs_names)

condition = cell_subset.obs["condition"].astype(str).values
# sample = cell_subset.obs["sample"].astype(str).values

print(f"data shape: {data.shape}")
assert data.shape == (len(genes), len(cells))


In [ ]:
%%R -i data -i genes -i cells -i condition -o deg_wilcoxon

library(Seurat)
library(Matrix)
library(dplyr)

# 1. Construct matrix
rownames(data) <- genes
colnames(data) <- cells
data <- as(data, "dgCMatrix")

# 2. Create Seurat object
seu <- CreateSeuratObject(counts = data)

# 3. Add metadata
seu$condition <- condition

# 4. Normalize (强烈建议保持仅 LogNormalize)
seu <- NormalizeData(
    seu,
    normalization.method = "LogNormalize",
    scale.factor = 10000
)

# 5. Set identities
Idents(seu) <- "condition"

# 6. Wilcoxon DEG (DCZ vs VEH)
deg_wilcoxon <- FindMarkers(
  seu,
  ident.1 = "nonplace",
  ident.2 = "silent",
  test.use = "wilcox",
  min.pct = 0.01,
  logfc.threshold = 0.0
)

head(deg_wilcoxon, 10)



In [ ]:
deg_wilcoxon.head(20)

In [ ]:
# -----------------------------------------------------------
# Save Wilcoxon DEG results
# -----------------------------------------------------------

output_filename = f"nonplace_silent_deg_wilcoxon_CA1ProsOnly_all_{today}.csv"
STORE_DIR = r"F:\Data\snRNAseq\PlaceCell\Results\DEGs\FindMarkers_wilcoxon"
output_path = os.path.join(STORE_DIR, output_filename)

# Export DEG table
deg_wilcoxon = deg_wilcoxon
deg_wilcoxon.to_csv(output_path, index=True)

print(f"✓ Wilcoxon DEG results saved:")
print(f"   → {output_path}")
print(f"   Genes detected: {deg_wilcoxon.shape[0]}")


Visualization of overlap genes between Wilcoxon and MAST

In [ ]:
# input_path = r"F:\Data\snRNAseq\PlaceCell\Results\DEGs"
# df_ns = pd.read_csv(os.path.join(input_path,"nonplace_silent_deg_wilcoxon_20260127.csv"),index_col=0)
# df_ps = pd.read_csv(os.path.join(input_path,"place_silent_deg_wilcoxon_20260127.csv"),index_col=0)
# df_pn = pd.read_csv(os.path.join(input_path,"place_nonplace_deg_wilcoxon_20260127.csv"),index_col=0)

In [ ]:
# set_ns = set(df_ns.index)
# set_ps = set(df_ps.index)
# set_pn = set(df_pn.index)

In [ ]:
# from matplotlib_venn import venn3
# import matplotlib.pyplot as plt

# plt.figure(figsize=(6,6))

# venn3(
#     [set_ns, set_ps, set_pn],
#     set_labels=("NonPlace vs Silent",
#                 "Place vs Silent",
#                 "Place vs NonPlace")
# )

# plt.title("DEG Overlap (wilcoxon)")
# plt.tight_layout()
# plt.show()


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)   # A higher target_sum enhances visual contrast

# Log-transform
sc.pp.log1p(adata)

In [ ]:
adata.obs['all_cells'] = 'all'

In [ ]:
deg_obj=ov.single.DEG(
    adata,
    condition='condition',
    ctrl_group='silent',
    test_group='nonplace',
    method='wilcoxon',
)
deg_obj.run(
    celltype_key='all_cells',
    celltype_group=['all']
)


In [ ]:
res_wilcoxon=deg_obj.get_results()
res_wilcoxon.head(20)

In [ ]:
# -----------------------------------------------------------
# Save MAST DEG results
# -----------------------------------------------------------

output_filename = f"nonplace_silent_deg_ov_wilcoxon_{today}.csv"
STORE_DIR = r"F:\Data\snRNAseq\PlaceCell\Results\DEGs\Ov_wilcoxon"
output_path = os.path.join(STORE_DIR, output_filename)

res_wilcoxon = res_wilcoxon
# Export DEG table
res_wilcoxon.to_csv(output_path, index=True)

print(f"✓ DEG results saved:")
print(f"   → {output_path}")
print(f"   Genes detected: {res_wilcoxon.shape[0]}")


In [ ]:
sc.pl.violin(
    adata,
    keys=['Stmn4','Gm42047','Dlgap1','Gpr85'],
    groupby='condition'
)

In [ ]:
from scipy.sparse import csr_matrix

# 确保是 CSR
if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)


In [ ]:
deg_obj=ov.single.DEG(
    adata,
    condition='condition',
    ctrl_group='nonplace',
    test_group='place',
    method='memento-de',
)
deg_obj.run(
    celltype_key='all_cells',
    celltype_group=['all'],
    capture_rate=0.07, 
    num_cpus=12,
    num_boot=5000
)


In [ ]:
res_memento=deg_obj.get_results()
res_memento.query('dv_coef > 1 & de_coef > 0').sort_values('de_pval').head(20)